# ArchAI demo – Stapfer-Enquête, Federal Consultations and the Archive Database
Runs on the shipped `data/out` (no raw dumps needed). Every output cites its archival source.
Apertus is used when `APERTUS_API_KEY` is set (env or `~/.config/archai/apertus.env`); otherwise answers are extractive.

In [1]:
import os, sys, json
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, 'src')
m = json.load(open('data/out/metrics.json'))
{k: m[k] for k in ('bern_reference', 'export')}

{'bern_reference': {'scope': 'University of Bern reference export: 14801 rows over 2245 sheets (12047 question answers, 2754 sheet-level fields not compared)',
  'sheets_found': 1.0,
  'answers_matched': 1.0,
  'signature_exact_match': 1.0,
  'text_similarity_median': 1.0,
  'text_exact_after_normalisation': 0.8169,
  'text_similarity_ge_0.95': 0.9982,
  'text_similarity_lt_0.80': 0.0002,
  'normalisation': 'lowercase, HTML entities decoded, page markers / gap marks / dashes removed, whitespace collapsed',
  'note': 'carried over from the last run with the Uni Bern reference file'},
 'export': {'records': 2819,
  'schema_invalid': 0,
  'apertus_merged': 2261,
  'scope': 'unified records.jsonl (Stapfer documents + consultations)'}}

## 1. One unified, provenance-preserving record

In [2]:
recs = [json.loads(l) for l in open('data/out/records.jsonl', encoding='utf-8')]
r = next(x for x in recs if x['provenance'].get('archive_signature', '').startswith('BAR B0 1000/1483, Nr. 1421'))
print(json.dumps({k: r[k] for k in ('record_id', 'title', 'dates', 'places', 'provenance', 'quality')}, ensure_ascii=False, indent=1, default=str)[:2500])

{
 "record_id": "stapfer:transkription:16",
 "title": "Stapfer-Enquête 1799 – Schule Hörnli",
 "dates": [],
 "places": [
  {
   "place_id": "stapfer:ort:214",
   "name": "Hörnli",
   "municipality_today": "Fischenthal",
   "lat": 47.370766,
   "lon": 8.941213,
   "source": "database"
  }
 ],
 "provenance": {
  "archive_signature": "BAR B0 1000/1483, Nr. 1421, fol. 228-229v",
  "repository": "Bundesarchiv Bern",
  "is_original": true,
  "facsimile_file": "16BAR_B0_10001483_Nr_1421_fol_228-229v.pdf",
  "viewer_url": "https://www.stapferenquete.ch/db/transkriptions/view/16",
  "source_dataset": "https://opendata.swiss/de/dataset/stapfer-enquete-schule-in-der-schweiz-1799",
  "source_tables": [
   "t_transkription",
   "t_antwort",
   "t_frage",
   "t_schule",
   "t_ort",
   "t_lehrer"
  ]
 },
 "quality": {
  "handwritten": true,
  "uncertain_readings": 1,
  "markup_errors_in_source": 0,
  "pages_in_transcription": 2,
  "legibility_rule": "partly_illegible",
  "legibility_rule_definition":

## 2. Deterministic tools (no model needed)

In [3]:
from archai import tools as T
print(T.normalize_dates('Fischenthal den 16.ten Hornung 1799'))
print(T.lookup_place('Hörndli'))
print(T.extract_quantities('Knaben 18. Mädchen 22. An Geld fl. 70 und 23 xr'))
print(T.lookup_archive_unit('E6100C#1998/106#1039*'))

[{'surface': '16.ten Hornung 1799', 'iso': '1799-02-16', 'year_inferred': False}]
[{'place_id': 'stapfer:ort:214', 'name_1799': 'Hörnli', 'municipality_today': 'Fischenthal', 'district_today': 'Hinwil', 'lat': 47.370766, 'lon': 8.941213, 'match': 'fuzzy'}]
{'pupils': [{'sex': 'male', 'count': 18, 'surface': 'Knaben 18'}, {'sex': 'female', 'count': 22, 'surface': 'Mädchen 22'}], 'money': [{'amount': 70.0, 'unit': 'fl', 'surface': 'fl. 70'}, {'amount': 23.0, 'unit': 'xr', 'surface': '23 xr'}]}
{'found': False, 'signature': 'E6100C#1998/106#1039*', 'reason': 'Archive Database not available locally (run the ingest stage)'}


## 3. Apertus tool-agent trace (plan → tools → compose → validate)

In [4]:
tr = json.loads(open('data/out/agent_traces.jsonl', encoding='utf-8').readline())
print(tr['record'], tr['model'], tr['prompt_version'])
for s in tr['steps']:
    print('--', s['step'], json.dumps(s.get('output') or s.get('results') or s.get('result'), ensure_ascii=False, default=str)[:400])

stapfer:transkription:2014 apertus-v1-5-70b agent-v3
-- plan {"calls": [], "reason": "No date or place extraction needed from provided text"}
-- tools [{"tool": "normalize_dates", "args": {"text": "Über die von der Regierung;\nbetreffend von dem Zustand der Schule an mich\neinges"}, "result": [], "origin": "baseline"}, {"tool": "extract_quantities", "args": {"text": "Über die von der Regierung;\nbetreffend von dem Zustand der Schule an mich\neinges"}, "result": {"pupils": [], "money": []}, "origin": "baseline"}, {"tool": "lookup_place", "args": 
-- compose {"title": "Antworten der Regierung auf Anfragen zum Zustand der Schule in Arth", "summary_en": "A document from the Helvetic Archive (B0 1000/1483, no. 1465, folios 81-82v) provides administrative details about the village of Arth, its district, municipality, and canton. It also describes the location of nearby settlements like Oberarth, Goldau, Walchwil, and Immensee relative to Arth.", "persons"
-- validate [{"entity": "Arth", "gro

## 4. Cited Q&A (rule router → hybrid retrieval → Apertus with [n] citations, extractive fallback)

In [5]:
from archai import qa, llm
use_llm = llm.KEY not in ('', 'none')
idx = qa.Index()
for q in ['Wie viele Kinder besuchten die Schule in Hörndli?', 'Quel est le salaire du régent à Lausanne?',
          'Which consultation dealt with Arbeitsvermittlung und Personalverleih?']:
    a = qa.answer(idx, q, use_llm=use_llm, rewrite='apertus')
    print('Q:', q, '\nA:', a['answer'][:500])
    for i, s in enumerate(a['sources'][:3], 1):
        print(f'  [{i}]', s['cite'], s.get('url', ''))
    print()

Q: Wie viele Kinder besuchten die Schule in Hörndli? 
A: [1] Hörnli (Lehrer: Jahannis Diener) 1799. Frage III.12 (Schulkinder. Wie viele Kinder besuchen überhaupt die Schule?): Zu Sommers zeit 24. Knaben 16. Töchteren.
[2] Hörnli (Lehrer: Jahannis Diener) 1799. Frage III.12.b (Im Sommer. (Knaben/Mädchen)): Zu Sommers zeit 24. Knaben 16. Töchteren.
  [1] BAR B0 1000/1483, Nr. 1421, fol. 228-229v, S. 2, Frage III.12 https://www.stapferenquete.ch/db/transkriptions/view/16
  [2] BAR B0 1000/1483, Nr. 1421, fol. 228-229v, S. 2, Frage III.12.b https://www.stapferenquete.ch/db/transkriptions/view/16

Q: Quel est le salaire du régent à Lausanne? 
A: [1] Lausanne, l'Hôpital (Lehrer: Amaron) 1799. Frage IV.16.A (En argent, bled, vin ou bois.): 76 L. deux sacs de Fromant soit 16 Quarteron
deux sacs Messel soit 10 quarteron payé par le Citoyen
Receveur d'es Pauvres de Lausanne plus 50 L.
deux Toises soit quatre char bois de Sapin du
Conseil de la Commu
[2] Lausanne, Le Bourg (Lehrer: Susanne Cathe

Q: Which consultation dealt with Arbeitsvermittlung und Personalverleih? 
A: [1] Bundesgesetz über die Arbeitsvermittlung und den Personalverleih (Bundesgesetz, EVD, 1984); Vernehmlassung 1984-01-31 bis 1984-05-31; Adressaten: . Dossier E6100C#1998/106#1039*: Entwurf des BG über die Arbeitsvermittlung und den Personalverleih - BRB 31.10.84 - Revision des Arbeitsvermittlungsgeset
[2] Revision des Arbeitsvermittlungsgesetzes vom 22.6.1951; Vorentwurf eines BG über die Arbeitsvermittlung und den Personalverleih (Bundesgesetz, EVD, 1983); Vernehmlassung 1983-12-22 bis 1984-0
  [1] E6100C#1998/106#1039* https://www.recherche.bar.admin.ch/recherche/#/de/archiv/einheit/2495973
  [2] Metadatenbank Vernehmlassungen Nr. 417 https://opendata.swiss/de/dataset/federal-consultations-1960-1991
  [3] Metadatenbank Vernehmlassungen Nr. 381 https://opendata.swiss/de/dataset/federal-consultations-1960-1991



## 5. Metrics (each block states its scope; model numbers are sample measurements, not overall accuracy)

In [6]:
print(json.dumps({k: m[k] for k in ('apertus_enrichment', 'apertus_agent', 'linking')}, indent=1, ensure_ascii=False, default=str)[:4000])

{
 "apertus_enrichment": {
  "apertus_enrichment.apertus-v1-5-70b.jsonl": {
   "scope": "apertus-v1-5-70b on a random sample (seed 7) of 2250 Stapfer documents, input truncated to 1800 chars",
   "json_valid_rate": 0.996,
   "teacher_surname_recall_vs_db": 0.974,
   "teacher_names_scored": 1064,
   "school_place_recall_vs_db": 0.921,
   "places_scored": 1303,
   "unsupported_entity_rate": 0.011,
   "unsupported_entity_definition": "person/place strings whose tokens do not all occur in the input text (proxy for hallucination; also catches normalised spellings)",
   "entities_checked": 20406,
   "legibility_flag_on_docs_with_uncertain_readings": {
    "good": 176,
    "partly_illegible": 28,
    "NaN": 1
   },
   "seconds_per_doc_median": 3.8,
   "note": "gold = hand-entered teacher/place tables of the Uni Bern Stapfer database (independent of the transcription text)"
  },
  "apertus_enrichment.apertus-v1-5-8b.jsonl": {
   "scope": "apertus-v1-5-8b on a random sample (seed 7) of 60 Stapf

## 6. Does Apertus query rewriting find the right sources? (`archai.qa_eval`)
Rules-only router vs. Apertus rewriting on 230 questions with known gold passages. Paraphrases were written to avoid the rule keywords, so rules-only scores on them are a floor.

In [7]:
e = m['qa_retrieval']
print(e['scope'])
for mode in ('off', 'apertus'):
    for g in ('stapfer_rules_wording', 'stapfer_paraphrased', 'consultations_paraphrased'):
        x = e['results'][mode][g]
        print(f"{mode:8} {g:27} top source right {x['hit_at_1']:.1%} · right source among 6 {x['hit_at_6']:.1%} · {x['seconds_median']} s")

230 questions: 216 templated Stapfer questions (9 sampled places, seed 3; 6 intents; 1 rules-wording + 3 paraphrases each, DE/FR/EN) + 14 hand-written consultation paraphrases
off      stapfer_rules_wording       top source right 100.0% · right source among 6 100.0% · 0.11 s
off      stapfer_paraphrased         top source right 8.6% · right source among 6 25.9% · 0.09 s
off      consultations_paraphrased   top source right 7.1% · right source among 6 14.3% · 0.09 s
apertus  stapfer_rules_wording       top source right 100.0% · right source among 6 100.0% · 1.2 s
apertus  stapfer_paraphrased         top source right 100.0% · right source among 6 100.0% · 1.17 s
apertus  consultations_paraphrased   top source right 21.4% · right source among 6 57.1% · 1.33 s


## 7. Number check on Apertus answers
Every number a sentence states must occur in the sources it cites (digits or French/German number words), otherwise the answer is replaced by the cited passages.

In [8]:
from archai.qa import check_numbers
hits = [{'cite': 'BAR B0 1000/1483, Nr. 1442, fol. 222-223v, S. 4', 'text': "l'Hôpital: 76 L. deux sacs de Fromant"},
        {'cite': 'BAR B0 1000/1483, Nr. 1442, fol. 220-221v, S. 4', 'text': 'Saint Laurent: deux cent Livres, bois environ 24 L.'}]
print(check_numbers('[1] Saint Laurent: deux cent Livres, bois environ 24 L.', hits, 'Salaire à Lausanne?'))  # wrong citation -> caught
print(check_numbers('[2] Saint Laurent: 24 L.', hits, 'Salaire à Lausanne?'))  # correct -> []
print({k: m['qa_answer_check'][k] for k in ('answers', 'accepted', 'rejected_number_not_in_cited_source', 'review')})

[{'number': '24', 'cited': [1], 'sentence': '[1] Saint Laurent: deux cent Livres, bois environ 24 L.'}]
[]
{'answers': 230, 'accepted': 185, 'rejected_number_not_in_cited_source': 6, 'review': 'all 6 number rejections checked by hand: 4 computed or inferred numbers (sums such as 17 + 22 = 39, a feast day read as a date), 2 numbers cited to the wrong source'}


## 8. Onboarding a new dataset (`archai.onboard`)
Any CSV/JSON/XLSX table → the same record schema, with provenance, privacy screen and a mapping that Apertus proposes and code validates. Here: the consultations table onboarded generically, compared with the hand-built records.

In [9]:
import pathlib
for ds in ('test-consultations-heuristic', 'test-consultations-apertus'):
    d = pathlib.Path('data/onboard') / ds
    if d.exists():
        r, cmp = json.load(open(d / 'report.json')), json.load(open(d / 'comparison.json'))
        print(ds, '| mapping:', r['mapping_source'], '| records:', r['records'], '| invalid:', r['schema_invalid'],
              '| titles:', cmp['title_exact'], '| dates:', cmp['reference_dates_found'], '| department:', cmp['lead_organisation_found'])
        print('   notes:', r['mapping_notes'])

test-consultations-heuristic | mapping: heuristic | records: 569 | invalid: 0 | titles: 1.0 | dates: 1.0 | department: 0.995
   notes: []
test-consultations-apertus | mapping: apertus | records: 569 | invalid: 0 | titles: 1.0 | dates: 1.0 | department: 0.995
   notes: ["title: 'doc_id' dropped (not readable text (an id or code))", "signature: 'federal_gazette_ref' dropped (values do not look like archive signatures)", 'title: Apertus proposal empty after validation — heuristic used']


In [10]:
# a messy file: Windows-1252, semicolons, LV95 coordinates, personal data -> flagged and masked
import tempfile
from archai import onboard
messy = 'Nr;Bezeichnung;Beschreibung;Datum;Ort;E;N\n1;Schulhaus Hörnli;Kontakt hans.muster@example.ch, Tel. 031 123 45 67;12.03.1962;Hörnli;2707000;1240000\n'
tmp = pathlib.Path(tempfile.mkdtemp()); (tmp / 'messy.csv').write_bytes(messy.encode('cp1252'))
onboard.BASE = tmp / 'onboard'
onboard.run(str(tmp / 'messy.csv'), 'messy', 'https://opendata.swiss/de/dataset/example', 'CC BY 4.0', None, True, 0, False, 'acquire', False)
rec = json.loads(open(tmp / 'onboard/messy/records.jsonl').readline())
print(rec['title'], rec['dates'], rec['places'][-1], rec['quality']['privacy_flags'], rec['text'], sep='\n')

{
 "dataset": "messy",
 "scope": "messy.csv (1 rows, sha256 641d0e5f715c…)",
 "source_url": "https://opendata.swiss/de/dataset/example",
 "licence": "CC BY 4.0",
 "records": 1,
 "schema_invalid": 0,
 "mapping_source": "heuristic",
 "mapping_notes": [],
 "apertus_mapping": null,
 "coverage": {
  "title": 1.0,
  "text": 1.0,
  "dates": 1.0,
  "places": 1.0,
  "places_geocoded": 1.0,
  "persons": 0.0,
  "archive_signature": 0.0
 },
 "records_with_privacy_flags": {
  "email": 1,
  "phone": 1
 },
 "pii_masked": true,
 "enriched": 0,
 "published_to_qa": false,
 "seconds": 0.1
}
Schulhaus Hörnli
[{'iso': '1962-03-12', 'surface': '12.03.1962', 'source': 'column:Datum'}]
{'place_id': None, 'name': None, 'lat': 47.302201, 'lon': 8.853519, 'source': 'columns:E,N'}
{'email': 2, 'phone': 2}
Beschreibung: Kontakt [REDACTED:email], Tel. [REDACTED:phone]
